In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import faiss
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
import spacy
from collections import Counter
import pickle
import json
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from transformers import AutoTokenizer, AutoModel
import torch
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA

In [219]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords



incidents_df = pd.read_csv('incidents_critique.csv', encoding='utf-8')

incidents_df = incidents_df.drop_duplicates(subset=["Description"]).reset_index(drop=True)

# size of the dataset


incidents_df.head()

,ID,Date,Description,Priorite
0,INC00000001,2023-01-15,Blocage validation virement SWIFT - client cor...,CRITIQUE
1,INC00000002,2023-02-28,Panne générale système de paiement instantané ...,CRITIQUE
2,INC00000003,2023-03-12,Incident SAB : échec validation multiple compt...,ELEVE
3,INC00000004,2023-04-08,Erreur synchronisation base données clients - ...,ELEVE
4,INC00000005,2023-05-22,Latence anormale système monétique - ralentiss...,ELEVE


In [220]:
print(f"Number of incidents: {len(incidents_df)}")

Number of incidents: 568


In [143]:
vectorizer = {}

In [222]:

try:
    stop_words = set(stopwords.words('french'))
except:
    nltk.download('stopwords')
    nltk.download('punkt')
    stop_words = set(stopwords.words('french'))

vectorizer['tfidf'] = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    stop_words=list(stop_words),
    # min_df=5,
    # max_df = 0.7,
    sublinear_tf=True
)

# Transformation des descriptions
processed_texts = incidents_df['Description'].fillna('').astype(str)
tfidf_matrix = vectorizer['tfidf'].fit_transform(processed_texts)

# Calcul des similarités
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"✅ TF-IDF entraîné sur {tfidf_matrix.shape[0]} incidents, {tfidf_matrix.shape[1]} features.")

✅ TF-IDF entraîné sur 568 incidents, 3408 features.


In [223]:
incident_embeddings = {}
similarity_matrices = {}
faiss_indices = {}

In [224]:
def fit_transform_tfidf( incidents_df):
        """Entraînement et transformation TF-IDF"""
        print("📊 Entraînement du modèle TF-IDF...")
        
        # vectorizer['tfidf'] = TfidfVectorizer(
        #         max_features=10000,
        #         ngram_range=(1, 3),
        #         stop_words=list(stop_words),
        #         min_df=2,
        #         max_df=0.85,
        #         sublinear_tf=True
        # )
        # Préprocessing
        tfidf_matrix = vectorizer['tfidf'].fit_transform(incidents_df['Description'])
        
        # Calcul de la matrice de similarité
        similarity_matrix = cosine_similarity(tfidf_matrix)
        
        incident_embeddings['tfidf'] = tfidf_matrix
        similarity_matrices['tfidf'] = similarity_matrix
        
        print(f"✅ TF-IDF: {tfidf_matrix.shape[0]} incidents, {tfidf_matrix.shape[1]} features")
        return tfidf_matrix

In [225]:
bert_model = SentenceTransformer('distiluse-base-multilingual-cased')

In [226]:
def fit_transform_bert( incidents_df):
        
        """Entraînement et transformation Sentence-BERT"""
        print("🧠 Génération des embeddings Sentence-BERT...")
        
        descriptions = incidents_df['Description'].fillna('').tolist()

        # 2️⃣ Génération des embeddings avec Sentence-BERT
        embeddings = bert_model.encode(
                descriptions,
                batch_size=32,
                show_progress_bar=True,
                normalize_embeddings=True  # Normalisation pour FAISS
        )

        # 3️⃣ Calcul de la matrice de similarité
        similarity_matrix = cosine_similarity(embeddings)

        # 4️⃣ Stockage des résultats
        incident_embeddings['bert'] = embeddings
        similarity_matrices['bert'] = similarity_matrix

        # 5️⃣ Création et remplissage de l'index FAISS
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(embeddings.astype('float32'))

        faiss_indices['bert'] = index

        print(f"✅ BERT: {embeddings.shape[0]} incidents, {embeddings.shape[1]}D embeddings")
        return embeddings

In [227]:
def create_hybrid_similarity( weights={'tfidf': 0.3, 'sbert': 0.7}):
        """Création d'une matrice de similarité hybride"""
        print("🔄 Création de la similarité hybride...")
        
        hybrid_matrix = np.zeros_like(similarity_matrices['tfidf'])
        
        for model_name, weight in weights.items():
            if model_name in similarity_matrices:
                hybrid_matrix += weight * similarity_matrices[model_name]
        
        similarity_matrices['hybrid'] = hybrid_matrix
        print(f"✅ Matrice hybride créée avec les poids: {weights}")
        
        return hybrid_matrix

In [151]:
similarity_matrices

{}

In [228]:
technical_weights = {
    'cft': 0.4,
    'autosys': 0.4,
    'ksh': 0.4,
    'batch': 0.3,
    'job': 0.2,
    'datalake': 0.3,
    'sap': 0.0,
    'cliker': 0.4,
    'compte': 0.3,
    'swift': 0.5,
    'virement': 0.4,
    'fixing': 0.4,
    'paiement': 0.3,
    'bloqué': 0.4,
    'carte': 0.4,
    'crm': 0.3,
    'fixing': 0.3,
    'oracle': 0.2,
    'incident': 0.0, 
    'serveur': 0.0,
    'client': 0.0
}

In [229]:
def compute_technical_score(text):
    score = 0.0
    text_lower = text.lower()
    for term, weight in technical_weights.items():
        if term in text_lower:
            score += weight
    return score

In [232]:
fit_transform_bert(incidents_df)

create_hybrid_similarity()


🧠 Génération des embeddings Sentence-BERT...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

✅ BERT: 568 incidents, 512D embeddings
🔄 Création de la similarité hybride...
✅ Matrice hybride créée avec les poids: {'tfidf': 0.3, 'sbert': 0.7}


array([[0.3       , 0.        , 0.00698565, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.3       , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.00698565, 0.        , 0.3       , ..., 0.        , 0.        ,
        0.0034029 ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.3       , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.3       ,
        0.        ],
       [0.        , 0.        , 0.0034029 , ..., 0.        , 0.        ,
        0.3       ]])

In [243]:
all_incidents_vectors = fit_transform_tfidf(incidents_df)

📊 Entraînement du modèle TF-IDF...
✅ TF-IDF: 568 incidents, 3408 features


In [159]:
create_hybrid_similarity()

🔄 Création de la similarité hybride...
✅ Matrice hybride créée avec les poids: {'tfidf': 0.3, 'sbert': 0.7}


array([[0.3       , 0.06352564, 0.09973762, ..., 0.11563345, 0.09442214,
        0.        ],
       [0.06352564, 0.3       , 0.08443739, ..., 0.        , 0.03730059,
        0.        ],
       [0.09973762, 0.08443739, 0.3       , ..., 0.        , 0.0376    ,
        0.        ],
       ...,
       [0.11563345, 0.        , 0.        , ..., 0.3       , 0.09272653,
        0.        ],
       [0.09442214, 0.03730059, 0.0376    , ..., 0.09272653, 0.3       ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.3       ]])

In [244]:
def search_similar_incidents(query_text, top_k=5, threshold=0.2):
    print(f"🔎 Requête : {query_text}")
    
    # 1️⃣ Vectoriser la requête
    query_vector = vectorizer['tfidf'].transform([query_text])
    print("✅ Vecteur TF-IDF de la requête généré. Shape:", query_vector.shape)
    # all_incidents_vectors = fit_transform_tfidf(incidents_df)
    # print(f"✅ Vecteurs BERT des incidents générés. Shape: {all_incidents_vectors.shape}")
    
    
    # 2️⃣ Calcul des similarités brutes
    similarities = cosine_similarity(query_vector, all_incidents_vectors).flatten()
    print(f" Similarités brutes (avant pondération) :")
    print(similarities)
    
    # 3️⃣ Calcul du score technique (bonus)
    query_score = compute_technical_score(query_text)
    print(f"🚀 Score pondéré de la requête (mots techniques) : {query_score:.3f}")
    
    boosted_similarities = []
    for idx, sim in enumerate(similarities):
        boost = compute_technical_score(incidents_df.iloc[idx]['Description'])
        final_score = sim + 0.2 * (query_score + boost)
        boosted_similarities.append((idx, final_score))
        print(f"Incident {idx} | TF-IDF : {sim:.3f} | Bonus : {boost:.3f} | Score final : {final_score:.3f}")
    
    # 4️⃣ Tri et filtrage
    boosted_similarities = sorted(boosted_similarities, key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, score in boosted_similarities:
        if score >= threshold:
            results.append((idx, score))
            if len(results) >= top_k:
                break
    
    print("\n🎯 Résultats finaux :")
    for i, (idx, score) in enumerate(results, 1):
        print(f"{i}. Incident {idx} | Score : {score:.3f}")
    
    return results

In [237]:
def search_similar_incidents_bert(query_text, top_k=5, threshold=0.2):
    print(f"🔎 Requête : {query_text}")
    
    # 1️⃣ Vectoriser la requête
    query_vector = fit_transform_bert(pd.DataFrame({'Description': [query_text]}))
    
    print("✅ Vecteur TF-IDF de la requête généré. Shape:", query_vector.shape)

    all_incidents_vectors = fit_transform_bert(incidents_df[['Description']])
    print(f"✅ Vecteurs BERT des incidents générés. Shape: {all_incidents_vectors.shape}")


    # if query_vector.shape[1] != all_incidents_vectors.shape[1]:
    #     raise ValueError(f"Incompatibilité des dimensions: Query={query_vector.shape[1]}, Incidents={all_incidents_vectors.shape[1]}")
    
    # 2️⃣ Calcul des similarités brutes
    similarities = cosine_similarity(query_vector, all_incidents_vectors).flatten()
    print(f" Similarités brutes (avant pondération) :")
    print(similarities)
    
    # 3️⃣ Calcul du score technique (bonus)
    query_score = compute_technical_score(query_text)
    print(f" Score pondéré de la requête (mots techniques) : {query_score:.3f}")
    
    boosted_similarities = []
    for idx, sim in enumerate(similarities):
        boost = compute_technical_score(incidents_df.iloc[idx]['Description'])
        final_score = sim + 0.2 * (query_score + boost)
        boosted_similarities.append((idx, final_score))
        print(f"Incident {idx} | TF-IDF : {sim:.3f} | Bonus : {boost:.3f} | Score final : {final_score:.3f}")
    
    # 4️⃣ Tri et filtrage
    boosted_similarities = sorted(boosted_similarities, key=lambda x: x[1], reverse=True)
    
    results = []
    for idx, score in boosted_similarities:
        if score >= threshold:
            results.append((idx, score))
            if len(results) >= top_k:
                break
    
    print("\n🎯 Résultats finaux :")
    for i, (idx, score) in enumerate(results, 1):
        print(f"{i}. Incident {idx} | Score : {score:.3f}")
    
    return results

In [ ]:

query_text = "Exécution d'une Requête SQL SAB des mouvements comptables"


# Rechercher les incidents similaires
similar_incidents = search_similar_incidents(query_text, top_k=5, threshold=0.2)


🔎 Requête : Exécution d'une Requête SQL SAB des mouvements comptables
✅ Vecteur TF-IDF de la requête généré. Shape: (1, 3408)
 Similarités brutes (avant pondération) :
[0.         0.         0.07512158 0.         0.         0.
 0.         0.         0.03610555 0.64425557 0.         0.
 0.         0.         0.05811443 0.03195278 0.         0.
 0.         0.07574507 0.03066145 0.         0.         0.07317386
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.02738054 0.07471745 0.         0.
 0.         0.         0.         0.04252636 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0

In [ ]:
#Resultats de la recherche de BERT



for idx, score in similar_incidents:
    print(f"Incident similaire : {incidents_df.iloc[idx]['Description']}")
    print(f"Score : {score:.3f}")
    print('-' * 40)



Incident similaire : Demande exécution requête SQL mouvements comptables du 08/10
Score : 0.645
----------------------------------------
Incident similaire : Incident SAB : échec validation multiple comptes entreprise - blocage opérations comptables
Score : 0.583
----------------------------------------
Incident similaire : Incident SAB : validation fusion comptes clients
Score : 0.561
----------------------------------------
Incident similaire : Blocage validation virement SWIFT - client corporate bloqué sur transfert international
Score : 0.539
----------------------------------------
Incident similaire : Validation virement BAS - demande transfert mail Casablanca
Score : 0.536
----------------------------------------


In [ ]:
#Resultats de la recherche de TF-IDF

for idx, score in similar_incidents:
    print(f"Incident similaire : {incidents_df.iloc[idx]['Description']}")
    print(f"Score : {score:.3f}")
    print('-' * 40)

Incident similaire : Demande exécution requête SQL mouvements comptables du 08/10
Score : 0.644
----------------------------------------
Incident similaire : Blocage validation virement SWIFT - client corporate bloqué sur transfert international
Score : 0.260
----------------------------------------


In [71]:
# Begin test

In [72]:
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time
import psutil
import gc
from collections import defaultdict
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

In [73]:
results = defaultdict(dict)
benchmarks = {}
evaluation_history = []

In [74]:
def _calculate_base_metrics( similarity_matrix):
        """Calcul des métriques de base"""
        # Extraction de la partie triangulaire supérieure
        mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
        similarities = similarity_matrix[mask]
        
        return {
            'mean_similarity': np.mean(similarities),
            'median_similarity': np.median(similarities),
            'std_similarity': np.std(similarities),
            'min_similarity': np.min(similarities),
            'max_similarity': np.max(similarities),
            'similarity_range': np.max(similarities) - np.min(similarities),
            'q1_similarity': np.percentile(similarities, 25),
            'q3_similarity': np.percentile(similarities, 75),
            'iqr_similarity': np.percentile(similarities, 75) - np.percentile(similarities, 25)
        }

In [75]:
def _calculate_skewness( data):
    """Calcul de l'asymétrie"""
    mean = np.mean(data)
    std = np.std(data)
    return np.mean(((data - mean) / std) ** 3)
    
def _calculate_kurtosis( data):
    """Calcul de l'aplatissement"""
    mean = np.mean(data)
    std = np.std(data)
    return np.mean(((data - mean) / std) ** 4) - 3

In [102]:
def _calculate_distribution_metrics(similarity_matrix):
        """Métriques de distribution des similarités"""
        mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
        similarities = similarity_matrix[mask]
        
        # Calcul des seuils de similarité
        high_sim = np.sum(similarities > 0.8) / len(similarities)
        medium_sim = np.sum((similarities > 0.4) & (similarities <= 0.8)) / len(similarities)
        low_sim = np.sum(similarities <= 0.4) / len(similarities)
        
        # Entropie de la distribution
        hist, _ = np.histogram(similarities, bins=10, density=True)
        hist = hist[hist > 0]  # Éviter log(0)
        entropy = -np.sum(hist * np.log2(hist + 1e-10))
        
        return {
            'high_similarity_ratio': high_sim,
            'medium_similarity_ratio': medium_sim,
            'low_similarity_ratio': low_sim,
            'distribution_entropy': entropy,
            'skewness': _calculate_skewness(similarities),
            'kurtosis': _calculate_kurtosis(similarities)
        }

In [76]:
def _evaluate_clustering_quality(similarity_matrix):
        """Évaluation de la qualité du clustering implicite"""
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score, calinski_harabasz_score
        
        # Conversion de la matrice de similarité en matrice de distance
        distance_matrix = 1 - similarity_matrix
        
        # Test de différents nombres de clusters
        silhouette_scores = []
        ch_scores = []
        
        for k in range(2, min(10, len(similarity_matrix) // 2)):
            try:
                kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
                labels = kmeans.fit_predict(distance_matrix)
                
                sil_score = silhouette_score(distance_matrix, labels, metric='precomputed')
                ch_score = calinski_harabasz_score(distance_matrix, labels)
                
                silhouette_scores.append(sil_score)
                ch_scores.append(ch_score)
            except:
                silhouette_scores.append(0)
                ch_scores.append(0)
        
        return {
            'best_silhouette_score': max(silhouette_scores) if silhouette_scores else 0,
            'best_ch_score': max(ch_scores) if ch_scores else 0,
            'optimal_clusters': np.argmax(silhouette_scores) + 2 if silhouette_scores else 2
        }

In [77]:
def _calculate_diversity_metrics( similarity_matrix):
        """Métriques de diversité"""
        # Calcul de la diversité moyenne
        mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
        similarities = similarity_matrix[mask]
        
        # Indice de Gini pour mesurer l'inégalité de distribution
        sorted_sims = np.sort(similarities)
        n = len(sorted_sims)
        index = np.arange(1, n + 1)
        gini = 2 * np.sum(index * sorted_sims) / (n * np.sum(sorted_sims)) - (n + 1) / n
        
        return {
            'diversity_index': 1 - np.mean(similarities),
            'gini_coefficient': gini,
            'unique_similarity_values': len(np.unique(similarities)),
            'similarity_variance': np.var(similarities)
        }

In [100]:
def evaluate_similarity_quality( similarity_system, incidents_df, ground_truth=None):
        """Évaluation complète de la qualité des similarités"""
        print("🎯 ÉVALUATION DE LA QUALITÉ DES SIMILARITÉS")
        print("=" * 50)
        
        results = {}
        
        for method in ['tfidf', 'sbert', 'hybrid']:
            if method in similarity_system['similarity_matrices']:
                print(f"\n📊 Évaluation de {method.upper()}...")


                
                # 1. Métriques de base
                similarity_matrix = similarity_system['similarity_matrices'][method]
                results[method] = _calculate_base_metrics(similarity_matrix)
                
                # 2. Métriques de distribution
                results[method].update(_calculate_distribution_metrics(similarity_matrix))
                
                # 3. Métriques de clustering
                results[method].update(_evaluate_clustering_quality(similarity_matrix))
                
                # 4. Métriques de diversité
                results[method].update(_calculate_diversity_metrics(similarity_matrix))

            else:
                print(f"❌ Méthode  non trouvée dans le système de similarité.")
                results[method] = {'error': 'Method not found'}
        
        results['quality'] = results
        return results

In [103]:
#je veux tester la fonction evaluate_similarity_quality
# use only the bert_model mais j'ai pas bert dans similarity_matrices

similarity_system = {
    'similarity_matrices': {
        'tfidf': similarity_matrix,
    }
}

evaluation_results = evaluate_similarity_quality(similarity_system, incidents_df)
print("\n📊 Résultats de l'évaluation de la qualité des similarités :")
print(evaluation_results)
# Affichage des résultats de l'évaluation
for method, metrics in evaluation_results.items():
    print(f"\n🔍 Méthode : {method}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}" if isinstance(value, float) else f"{metric}: {value}")

🎯 ÉVALUATION DE LA QUALITÉ DES SIMILARITÉS

📊 Évaluation de TFIDF...


c:\Users\MOHCINE_01\anaconda3\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


❌ Méthode  non trouvée dans le système de similarité.
❌ Méthode  non trouvée dans le système de similarité.

📊 Résultats de l'évaluation de la qualité des similarités :
{'tfidf': {'mean_similarity': 0.10592515323280798, 'median_similarity': 0.0, 'std_similarity': 0.14865516455216377, 'min_similarity': 0.0, 'max_similarity': 0.8937489068075748, 'similarity_range': 0.8937489068075748, 'q1_similarity': 0.0, 'q3_similarity': 0.15467755947218031, 'iqr_similarity': 0.15467755947218031, 'high_similarity_ratio': 0.0016130082795983189, 'medium_similarity_ratio': 0.05554342287253403, 'low_similarity_ratio': 0.9428435688478677, 'distribution_entropy': -16.875026255150548, 'skewness': 1.6532761130684348, 'kurtosis': 2.773529120289589, 'best_silhouette_score': 0, 'best_ch_score': 0, 'optimal_clusters': 2, 'diversity_index': 0.8940748467671921, 'gini_coefficient': 0.6822751053834146, 'unique_similarity_values': 115863, 'similarity_variance': 0.022098357948030885}, 'sbert': {'error': 'Method not foun

In [139]:
def _test_scalability( fit_transform_bertsimilarity_system, incidents_df, method):
        """Test de scalabilité avec différentes tailles de données"""
        scalability_results = {}
        
        sizes = [len(incidents_df) // 4, len(incidents_df) // 2, len(incidents_df)]
        times = []
        
        for size in sizes:
            if size > 0:
                sample_df = incidents_df.head(size)
                start_time = time.time()
                
                # Simulation du temps de traitement
                if method == 'tfidf':
                    _ = fit_transform_bert(
                        sample_df
                    )
                elif method == 'sbert':
                    _ = similarity_system.models['sbert'].encode(
                        sample_df['description'].fillna('').tolist()[:min(size, 50)]
                    )
                
                end_time = time.time()
                times.append(end_time - start_time)
        
        if len(times) > 1:
            # Estimation de la complexité (linéaire, quadratique, etc.)
            complexity_ratio = times[-1] / times[0] if times[0] > 0 else 1
            scalability_results['complexity_ratio'] = complexity_ratio
            scalability_results['scalability_score'] = 1 / complexity_ratio if complexity_ratio > 0 else 0
        
        scalability_results['processing_times'] = times
        return scalability_results

In [140]:
def _test_robustness( similarity_system, incidents_df, method):
        """Test de robustesse avec données bruitées"""
        robustness_results = {}
        
        try:
            # Test avec données manquantes
            corrupted_df = incidents_df.copy()
            corrupted_df.loc[corrupted_df.index[:len(corrupted_df)//4], 'description'] = ''
            
            start_time = time.time()
            # Test de recherche avec données corrompues
            for i in range(min(5, len(corrupted_df))):
                try:
                    similar_indices, scores = similarity_system.find_similar_incidents(
                        i, method=method, top_k=3
                    )
                except:
                    pass
            
            robustness_time = time.time() - start_time
            robustness_results['robustness_time'] = robustness_time
            robustness_results['handles_missing_data'] = True
            
        except Exception as e:
            robustness_results['robustness_time'] = float('inf')
            robustness_results['handles_missing_data'] = False
        
        return robustness_results

In [141]:
def benchmark_performance(similarity_system, incidents_df, iterations=5):
        """Benchmark de performance (vitesse, mémoire, scalabilité)"""
        print("\n⚡ BENCHMARK DE PERFORMANCE")
        print("=" * 50)
        
        results = {}
        
        for method in ['tfidf', 'sbert', 'hybrid']:
            if method in similarity_system['similarity_matrices']:
                print(f"\n🚀 Benchmark {method.upper()}...")
                
                results[method] = {}
                
                # 1. Test de vitesse de recherche
                search_times = []
                memory_usage = []
                
                for i in range(iterations - 1):
                    # Mesure de la mémoire avant
                    gc.collect()
                    mem_before = psutil.Process().memory_info().rss / 1024 / 1024  # MB
                    
                    # Test de vitesse
                    start_time = time.time()
                    
                    # Simulation de recherches multiples
                    # for query_idx in range(min(10, len(incidents_df))):
                    #     similar_indices, scores = search_similar_incidents(
                    #         query_idx,incidents_df
                    #     )

                    for i in range(min(10, len(incidents_df))):
                        query_text = incidents_df.iloc[i]['description']
                        similar_incidents = search_similar_incidents(query_text, top_k=5, threshold=0.2)

                    
                    end_time = time.time()
                    search_times.append((end_time - start_time) / 10)  # Temps moyen par recherche
                    
                    # Mesure de la mémoire après
                    mem_after = psutil.Process().memory_info().rss / 1024 / 1024  # MB
                    memory_usage.append(mem_after - mem_before)
                
                results[method]['avg_search_time'] = np.mean(search_times)
                results[method]['std_search_time'] = np.std(search_times)
                results[method]['avg_memory_usage'] = np.mean(memory_usage)
                results[method]['std_memory_usage'] = np.std(memory_usage)
                
                # 2. Test de scalabilité
                results[method].update(_test_scalability(similarity_system, incidents_df, method))
                
                # 3. Test de robustesse
                results[method].update(_test_robustness(similarity_system, incidents_df, method))
        
        results['performance'] = results
        return results

In [142]:
benchmark_results = benchmark_performance(similarity_system, incidents_df, iterations=3)


⚡ BENCHMARK DE PERFORMANCE

🚀 Benchmark TFIDF...
🔎 Requête : Le service informatique rencontrent une situation de panne critique sur le logiciel DBS.
✅ Vecteur TF-IDF de la requête généré.
📈 Similarités brutes (avant pondération) :
[1.         0.21175213 0.33245873 0.         0.         0.
 0.52987263 0.         0.         0.         0.         0.
 0.         0.15353755 0.         0.         0.         0.1234209
 0.66879093 0.11686663 0.         0.         0.24917226 0.38418511
 0.         0.39425987 0.72120666 0.29463707 0.         0.
 0.         0.53449889 0.         0.         0.         0.60196146
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.35524176 0.         0.13014112
 0.39395262 0.         0.39513545 0.         0.         0.
 0.46960291 0.         0.         0.         0.         0.
 0.6582453  0.31623938 0.38795391 0.15826628 0.         0.
 0.29591006 0.31144479 0.         0.         0.         0.
 0.         0.         0.   

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ BERT: 180 incidents, 512D embeddings
🧠 Génération des embeddings Sentence-BERT...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ BERT: 361 incidents, 512D embeddings
🧠 Génération des embeddings Sentence-BERT...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

✅ BERT: 723 incidents, 512D embeddings
